### Nexmark

In [ ]:
from pyflink.table import EnvironmentSettings, TableEnvironment
import os

from pathlib import Path

from pyflink.java_gateway import get_gateway
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import StreamTableEnvironment, ExplainDetail

gateway = get_gateway()
string_class = gateway.jvm.String
string_array = gateway.new_array(string_class, 0)
stream_env = gateway.jvm.org.apache.flink.streaming.api.environment.StreamExecutionEnvironment
j_stream_exection_environment = stream_env.createRemoteEnvironment(
    "localhost", 
    8081, 
    string_array
)

env = StreamExecutionEnvironment(j_stream_exection_environment)
env.set_parallelism(4)
env.enable_checkpointing(30000)
table_env = StreamTableEnvironment.create(env)
jar_path = Path("../flink-sql-connector-kafka-4.0.0-2.0.jar").resolve().as_uri()
table_env.get_config().set("pipeline.jars", jar_path)
t_env=table_env
current_dir = os.getcwd()

# Define the source table using DDL (update the file path as needed)
source_ddl = """
CREATE TABLE kafka (
    event_type int,
    person ROW<
        id  BIGINT,
        name  VARCHAR,
        emailAddress  VARCHAR,
        creditCard  VARCHAR,
        city  VARCHAR,
        state  VARCHAR,
        `dateTime` TIMESTAMP(3),
        extra  VARCHAR>,
    auction ROW<
        id  BIGINT,
        itemName  VARCHAR,
        description  VARCHAR,
        initialBid  BIGINT,
        reserve  BIGINT,
        `dateTime`  TIMESTAMP(3),
        expires  TIMESTAMP(3),
        seller  BIGINT,
        category  BIGINT,
        extra  VARCHAR>,
    bid ROW<
        auction  BIGINT,
        bidder  BIGINT,
        price  BIGINT,
        channel  VARCHAR,
        url  VARCHAR,
        `dateTime`  TIMESTAMP(3),
        extra  VARCHAR>,
    `dateTime` AS
        CASE
            WHEN event_type = 0 THEN person.`dateTime`
            WHEN event_type = 1 THEN auction.`dateTime`
            ELSE bid.`dateTime`
        END,
    WATERMARK FOR `dateTime` AS `dateTime` - INTERVAL '4' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'event-demo',
    'properties.bootstrap.servers' = 'kafka-service.kafka.svc.cluster.local:9092',
    'properties.group.id' = 'nexmark',
    'scan.startup.mode' = 'earliest-offset',
    'sink.partitioner' = 'round-robin',
    'format' = 'json'
);
"""
t_env.execute_sql(source_ddl)

# Define the sink table using DDL with the print connector for debugging/output
sink_ddl = """
CREATE TABLE nexmark_q11 (
  bidder BIGINT,
  bid_count BIGINT,
  starttime TIMESTAMP(3),
  endtime TIMESTAMP(3)
) WITH (
  'connector' = 'print'
);
"""
t_env.execute_sql(sink_ddl)

### do the query
query = """
INSERT INTO nexmark_q11
SELECT
    bidder,
    bid_count,
    starttime,
    endtime
FROM (
    SELECT
        bid.bidder,
        COUNT(*) as bid_count,
        SESSION_START(`dateTime`, INTERVAL '10' SECOND) as starttime,
        SESSION_END(`dateTime`, INTERVAL '10' SECOND) as endtime
    FROM kafka
    GROUP BY
        bid.bidder,
        SESSION(`dateTime`, INTERVAL '10' SECOND)
)
"""
t_env.execute_sql(query)